In [1]:
import pandas as pd
import numpy as np
import joblib


In [2]:
df = pd.read_csv("../data/gene_expression/biofilm_ml_features.csv")
print("Loaded:", df.shape)
df.head()


Loaded: (5983, 16)


,gene_symbol,biofilm_logFC,biofilm_padj,planktonic_logFC,planktonic_padj,biofilm_abs_logFC,planktonic_abs_logFC,biofilm_significant,planktonic_significant,logFC_interaction,abs_logFC_interaction,logFC_difference,abs_logFC_difference,logFC_ratio,abs_logFC_ratio,biofilm_label
0,PA14_22460,4.345,1.200000e-67,4.345,0.05,4.345,4.345,1,0,18.879025,18.879025,0.0,0.0,1.0,1.0,1
1,ppiA,4.151,3.000000e-63,4.151,0.05,4.151,4.151,1,0,17.230801,17.230801,0.0,0.0,1.0,1.0,1
2,PA14_22470,3.929,1.700000e-62,3.929,0.05,3.929,3.929,1,0,15.437041,15.437041,0.0,0.0,1.0,1.0,1
3,bapA,3.773,4.400000e-59,3.773,0.05,3.773,3.773,1,0,14.235529,14.235529,0.0,0.0,1.0,1.0,1
4,PA14_27070,3.263,1.600000e-46,3.263,0.05,3.263,3.263,1,0,10.647169,10.647169,0.0,0.0,1.0,1.0,1


In [3]:
try:
    model = joblib.load("../models/random_forest.pkl")
    model_name = "Random Forest"
except:
    model = joblib.load("../models/logistic_regression.pkl")
    model_name = "Logistic Regression"

print("Loaded model:", model_name)


Loaded model: Logistic Regression


In [4]:
feature_cols = [
    "biofilm_logFC", "biofilm_padj",
    "planktonic_logFC", "planktonic_padj",
    "biofilm_abs_logFC", "planktonic_abs_logFC",
    "biofilm_significant", "planktonic_significant",
    "logFC_interaction", "abs_logFC_interaction",
    "logFC_difference", "abs_logFC_difference",
    "logFC_ratio", "abs_logFC_ratio"
]


In [5]:
def predict_gene(gene_symbol):
    # Check if gene exists
    if gene_symbol not in df["gene_symbol"].values:
        return f"Gene '{gene_symbol}' not found in dataset."

    # Extract row
    row = df[df["gene_symbol"] == gene_symbol].iloc[0]

    # Extract features
    X_gene = row[feature_cols].values.reshape(1, -1)

    # Predict probability
    if model_name == "Random Forest":
        prob = model.predict_proba(X_gene)[0][1]
    else:
        prob = model.predict_proba(X_gene)[0][1]

    # Predict class
    pred_class = int(prob >= 0.5)

    # Build result dictionary
    result = {
        "gene_symbol": gene_symbol,
        "predicted_probability": prob,
        "predicted_class": pred_class,
        "true_label": int(row["biofilm_label"]),
        "feature_values": row[feature_cols].to_dict()
    }

    return result


In [6]:
test_gene = df["gene_symbol"].iloc[0]
predict_gene(test_gene)


{'gene_symbol': 'PA14_22460',
 'predicted_probability': np.float64(8.538409025353402e-05),
 'predicted_class': 0,
 'true_label': 1,
 'feature_values': {'biofilm_logFC': 4.345,
  'biofilm_padj': 1.2e-67,
  'planktonic_logFC': 4.345,
  'planktonic_padj': 0.05,
  'biofilm_abs_logFC': 4.345,
  'planktonic_abs_logFC': 4.345,
  'biofilm_significant': 1,
  'planktonic_significant': 0,
  'logFC_interaction': 18.879025,
  'abs_logFC_interaction': 18.879025,
  'logFC_difference': 0.0,
  'abs_logFC_difference': 0.0,
  'logFC_ratio': 0.9999997698504556,
  'abs_logFC_ratio': 0.9999997698504556}}

In [7]:
gene = input("Enter a gene symbol: ").strip()
prediction = predict_gene(gene)
prediction


Enter a gene symbol:  PA14_22460 ppiA lasR rpoS


"Gene 'PA14_22460 ppiA lasR rpoS' not found in dataset."

In [8]:
all_predictions = []

for gene in df["gene_symbol"].values:
    result = predict_gene(gene)
    if isinstance(result, dict):
        all_predictions.append(result)

pred_df = pd.DataFrame(all_predictions)
pred_df.to_csv("../data/gene_expression/biofilm_predictions.csv", index=False)

print("Saved gene-level predictions.")


Saved gene-level predictions.
